# L16 - Reti a più strati e retropropagazione

Eseguire le celle in ordine con `Maiusc + Invio`.

In [ ]:
# Funzione di controllo degli esercizi: eseguire questa cella, non modificarla.
def controlla(numero, prova):
    try:
        esito = bool(prova())
    except Exception as e:
        print(f"Esercizio {numero}: da rivedere ({type(e).__name__})")
        return
    print(f"Esercizio {numero}: " + ("corretto" if esito else "da rivedere"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoide(z):
    return 1 / (1 + np.exp(-z))

P = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)

## 1. Il calcolo in avanti (forward pass)

Una rete con uno strato nascosto: ingressi $x$, strato nascosto con pesi $W_1$ e bias $b_1$, uscita con pesi $W_2$ e bias $b_2$.

$$h = \sigma(W_1 x + b_1) \qquad \hat{y} = \sigma(W_2 h + b_2)$$

Con tutti gli esempi insieme (una riga per esempio) si usa `X @ W1.T` come nella lezione L11.

In [ ]:
def avanti(X, W1, b1, W2, b2):
    """Calcolo in avanti: restituisce le uscite dello strato nascosto e le previsioni."""
    H = sigmoide(X @ W1.T + b1)          # forma (esempi, neuroni nascosti)
    Y_prev = sigmoide(H @ W2.T + b2)     # forma (esempi, 1)
    return H, Y_prev

# Pesi scelti a mano: neurone nascosto 1 simile a OR, neurone 2 simile a NAND, uscita simile a AND.
# I pesi sono moltiplicati per 10 perché la sigmoide si avvicini al gradino.
W1 = 10 * np.array([[1, 1], [-1, -1]])
b1 = 10 * np.array([-0.5, 1.5])
W2 = 10 * np.array([[1, 1]])
b2 = 10 * np.array([-1.5])
H, Y_prev = avanti(P, W1, b1, W2, b2)
print(np.round(Y_prev.ravel(), 3))

## 2. Perché serve la non linearità

Senza funzione di attivazione, due strati equivalgono a uno solo: $W_2 (W_1 x + b_1) + b_2 = (W_2 W_1) x + (W_2 b_1 + b_2)$, cioè ancora una funzione lineare. La cella lo verifica numericamente.

In [ ]:
rng = np.random.default_rng(0)
A1, c1 = rng.normal(size=(3, 2)), rng.normal(size=3)
A2, c2 = rng.normal(size=(1, 3)), rng.normal(size=1)
x = np.array([0.7, -1.2])
due_strati = A2 @ (A1 @ x + c1) + c2
uno_strato = (A2 @ A1) @ x + (A2 @ c1 + c2)
print(due_strati, uno_strato)

## 3. Addestramento con la retropropagazione

La funzione `addestra_rete` addestra la rete con la discesa del gradiente (perdita MSE). Il gradiente si calcola all'indietro, dall'uscita verso l'ingresso, applicando la regola della catena:

1. calcolo in avanti: `H`, `Y_prev`
2. errore all'uscita, moltiplicato per la derivata della sigmoide: `d2`
3. gradienti di `W2`, `b2` a partire da `d2` e dalle uscite nascoste `H`
4. errore riportato allo strato nascosto attraverso `W2`, moltiplicato per la derivata della sigmoide: `d1`
5. gradienti di `W1`, `b1` a partire da `d1` e dagli ingressi `X`
6. aggiornamento di tutti i parametri

La derivata della sigmoide è $\sigma(z)(1 - \sigma(z))$ (lezione L6).

In [ ]:
def addestra_rete(X, y, nascosti=4, eta=2.0, epoche=5000, seme=0):
    """Rete con uno strato nascosto di neuroni sigmoide e un neurone di uscita sigmoide."""
    rng = np.random.default_rng(seme)
    W1 = rng.normal(0, 1, size=(nascosti, X.shape[1]))
    b1 = np.zeros(nascosti)
    W2 = rng.normal(0, 1, size=(1, nascosti))
    b2 = np.zeros(1)
    Y = y.reshape(-1, 1)                          # etichette come colonna
    perdite = []
    for epoca in range(epoche):
        # 1. calcolo in avanti
        H, Y_prev = avanti(X, W1, b1, W2, b2)
        perdite.append(((Y_prev - Y) ** 2).mean())
        # 2. errore all'uscita per la derivata della sigmoide
        d2 = 2 * (Y_prev - Y) / len(X) * Y_prev * (1 - Y_prev)
        # 3. gradienti dello strato di uscita
        g_W2 = d2.T @ H
        g_b2 = d2.sum(axis=0)
        # 4. errore riportato allo strato nascosto
        d1 = (d2 @ W2) * H * (1 - H)
        # 5. gradienti dello strato nascosto
        g_W1 = d1.T @ X
        g_b1 = d1.sum(axis=0)
        # 6. aggiornamento
        W2 = W2 - eta * g_W2
        b2 = b2 - eta * g_b2
        W1 = W1 - eta * g_W1
        b1 = b1 - eta * g_b1
    return W1, b1, W2, b2, perdite

W1, b1, W2, b2, perdite = addestra_rete(P, y_xor, nascosti=4)
print("uscite:", np.round(avanti(P, W1, b1, W2, b2)[1].ravel(), 3))
plt.plot(perdite)
plt.xlabel("epoca")
plt.ylabel("perdita MSE")
plt.title("Addestramento su XOR")
plt.show()

## 4. Un problema più difficile: le due lune

Due classi di punti a forma di mezzaluna, intrecciate: nessuna retta le separa.

In [ ]:
def lune(n=200, rumore=0.15, seme=0):
    """n punti in due mezzelune intrecciate, con etichette 0 e 1."""
    rng = np.random.default_rng(seme)
    m = n // 2
    t = rng.uniform(0, np.pi, m)
    a = np.column_stack([np.cos(t), np.sin(t)])
    b = np.column_stack([1 - np.cos(t), 0.5 - np.sin(t)])
    X = np.vstack([a, b]) + rng.normal(0, rumore, size=(2 * m, 2))
    y = np.concatenate([np.zeros(m), np.ones(m)])
    return X, y

def regione(X, y, parametri, titolo=""):
    """Disegna l'uscita della rete sul piano e i punti."""
    g1 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 150)
    g2 = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 150)
    G1, G2 = np.meshgrid(g1, g2)
    griglia = np.column_stack([G1.ravel(), G2.ravel()])
    uscita = avanti(griglia, *parametri)[1].reshape(G1.shape)
    plt.contourf(G1, G2, uscita, levels=20, cmap="Greys", alpha=0.6)
    plt.scatter(X[y == 0, 0], X[y == 0, 1], marker="o", s=15, label="classe 0")
    plt.scatter(X[y == 1, 0], X[y == 1, 1], marker="s", s=15, label="classe 1")
    plt.title(titolo)
    plt.legend()
    plt.show()

X, y = lune()
W1, b1, W2, b2, perdite = addestra_rete(X, y, nascosti=8)
previsioni = (avanti(X, W1, b1, W2, b2)[1].ravel() >= 0.5)
print("accuratezza:", (previsioni == y).mean())
regione(X, y, (W1, b1, W2, b2), "Rete con 8 neuroni nascosti")

`np.column_stack` affianca array come colonne; `ravel()` trasforma un array di qualsiasi forma in un array a una dimensione; `*parametri` passa gli elementi della tupla come argomenti separati.

## Esercizi

**Esercizio 1 (base).** Calcolare con `avanti` le uscite della rete con i pesi scelti a mano della sezione 1, arrotondarle a 0 o 1 (`>= 0.5`) e assegnare il risultato (array di interi) a `uscite_mano`. Verificare che siano quelle di XOR.

In [ ]:
W1m = 10 * np.array([[1, 1], [-1, -1]])
b1m = 10 * np.array([-0.5, 1.5])
W2m = 10 * np.array([[1, 1]])
b2m = 10 * np.array([-1.5])
uscite_mano = (avanti(P, W1m, b1m, W2m, b2m)[1].ravel() >= 0.5).astype(int)
print(uscite_mano)

In [ ]:
controlla(1, lambda: np.array_equal(uscite_mano, [0, 1, 1, 0]))

**Esercizio 2 (base).** Addestrare la rete su XOR con 2 neuroni nascosti per i semi da 0 a 4. Salvare nella lista `perdite_finali` l'ultimo valore della perdita per ciascun seme. Con alcuni semi la rete non impara: si ferma in un minimo locale.

In [ ]:
perdite_finali = []
for s in range(5):
    *_, p = addestra_rete(P, y_xor, nascosti=2, seme=s)
    perdite_finali.append(p[-1])
print(np.round(perdite_finali, 4))

In [ ]:
controlla(2, lambda: len(perdite_finali) == 5 and min(perdite_finali) < 0.01 and max(perdite_finali) > 0.1)

**Esercizio 3 (standard).** Sulle due lune, addestrare reti con 1, 2, 4 e 8 neuroni nascosti. Salvare l'accuratezza di ciascuna nel dizionario `acc_nascosti` con chiave il numero di neuroni e disegnare la regione di decisione della rete con 2 neuroni. Descrivere in una cella di testo come cambia la forma del confine.

In [ ]:
acc_nascosti = {}
for k in [1, 2, 4, 8]:
    parametri = addestra_rete(X, y, nascosti=k)[:4]
    acc_nascosti[k] = float(((avanti(X, *parametri)[1].ravel() >= 0.5) == y).mean())
    if k == 2:
        regione(X, y, parametri, "2 neuroni nascosti")
print(acc_nascosti)

In [ ]:
controlla(3, lambda: set(acc_nascosti) == {1, 2, 4, 8} and acc_nascosti[8] > 0.95 and acc_nascosti[1] < 0.95)

**Esercizio 4 (standard).** Effetto del tasso di apprendimento sulle due lune con 8 neuroni nascosti: addestrare con `eta` pari a 0.1, 2 e 20 per 5000 epoche e disegnare le tre curve della perdita nello stesso grafico. Salvare le perdite finali nel dizionario `perdita_eta`.

In [ ]:
perdita_eta = {}
for eta in [0.1, 2, 20]:
    *_, p = addestra_rete(X, y, nascosti=8, eta=eta)
    perdita_eta[eta] = float(p[-1])
    plt.plot(p, label=f"eta = {eta}")
plt.yscale("log")
plt.xlabel("epoca")
plt.ylabel("perdita")
plt.legend()
plt.show()
print(perdita_eta)

In [ ]:
controlla(4, lambda: set(perdita_eta) == {0.1, 2, 20} and perdita_eta[2] < perdita_eta[0.1])

**Esercizio 5 (approfondimento).** Verificare numericamente un gradiente della retropropagazione. Con la rete iniziale (seme 0, 4 neuroni nascosti, dati XOR) calcolare la derivata della perdita rispetto al peso `W1[0, 0]` in due modi: con il rapporto incrementale (variando solo quel peso di ±0.0001) e con le formule della sezione 3 (primo passo dell'addestramento). Assegnare i due valori a `g_numerico` e `g_formula`.

In [ ]:
rng = np.random.default_rng(0)
W1i = rng.normal(0, 1, size=(4, 2)); b1i = np.zeros(4)
W2i = rng.normal(0, 1, size=(1, 4)); b2i = np.zeros(1)
Y = y_xor.reshape(-1, 1)

def perdita_con(w00):
    W = W1i.copy(); W[0, 0] = w00
    return ((avanti(P, W, b1i, W2i, b2i)[1] - Y) ** 2).mean()

h = 1e-4
g_numerico = (perdita_con(W1i[0, 0] + h) - perdita_con(W1i[0, 0] - h)) / (2 * h)
H, Yp = avanti(P, W1i, b1i, W2i, b2i)
d2 = 2 * (Yp - Y) / len(P) * Yp * (1 - Yp)
d1 = (d2 @ W2i) * H * (1 - H)
g_formula = (d1.T @ P)[0, 0]
print(g_numerico, g_formula)

In [ ]:
controlla(5, lambda: np.isclose(g_numerico, g_formula, rtol=1e-4))